# Análisis de Métricas y Recomendaciones de Marketing

Este notebook realiza un análisis utilizando tres datasets:

* **visits:** Registra las visitas al sitio web (con columnas *Uid*, *Device*, *Start Ts*, *End Ts*, *Source Id*).
* **orders:** Registra los pedidos realizados (con columnas *Uid*, *Buy Ts*, *Revenue*).
* **costs:** Registra los gastos en publicidad (con columnas *source_id*, *dt*, *costs*).

Además, se genera un dataset **sintético** (con la misma estructura y columnas) para ilustrar cómo se pueden analizar datos simulados.

El proyecto abarca los siguientes pasos:

1. **Exploración y preparación de los datos.**
2. **Cálculo de métricas e informes para visitas, ventas y marketing.**
   - Visitas: Usuarios únicos diarios, semanales y mensuales, sesiones diarias, duración de las sesiones y frecuencia de retorno.
   - Ventas: Tiempo hasta la conversión (de registro a la primera compra), cantidad de pedidos, tamaño promedio de compra y Lifetime Value (LTV).
   - Marketing: Gasto total (y por fuente), costo de adquisición de clientes (CAC) y rentabilidad de las inversiones (ROMI). Se incluyen gráficos segmentados por dispositivo y fuente de anuncios.
3. **Conclusiones y recomendaciones.**

Las celdas de código y de texto están organizadas y documentadas para facilitar la comprensión del análisis.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import datetime

## 2. Carga de Datos Reales

A continuación se cargan los datasets reales proporcionados. Se asume que los archivos están disponibles en las rutas indicadas. Se convierten las columnas de fecha a objetos datetime para facilitar su manejo.

In [ ]:
# Cargar datasets reales
visits = pd.read_csv('/mnt/data/visits_log_us.csv', parse_dates=['Start Ts', 'End Ts'])
orders = pd.read_csv('/mnt/data/orders_log_us.csv', parse_dates=['Buy Ts'])
costs = pd.read_csv('/mnt/data/costs_us.csv', parse_dates=['dt'])

# Mostrar las primeras filas de cada dataset
print("Visitas (Datos reales)")
display(visits.head())

print("Orders (Datos reales)")
display(orders.head())

print("Costs (Datos reales)")
display(costs.head())

## 3. Generación de Dataset Sintético

A continuación se genera un conjunto de datos sintéticos con la misma estructura que los datasets reales. Esto nos permitirá simular y validar el análisis sin depender únicamente de los datos proporcionados.

**Estructura de los datos sintéticos:**

* **visits_synth:**
  - *Uid:* identificador único del usuario.
  - *Device:* Dispositivo del usuario (ej. 'desktop', 'mobile', 'tablet').
  - *Start Ts:* Fecha y hora de inicio de la sesión.
  - *End Ts:* Fecha y hora de término de la sesión.
  - *Source Id:* Identificador de la fuente de anuncios.

* **orders_synth:**
  - *Uid:* Identificador único del usuario.
  - *Buy Ts:* Fecha y hora de la compra.
  - *Revenue:* Ingreso generado por la orden.

* **costs_synth:**
  - *source_id:* Identificador de la fuente de anuncios.
  - *dt:* Fecha.
  - *costs:* Gasto realizado en esa fuente en el día.


In [ ]:
# Generar dataset sintético para 'visits'

np.random.seed(42)
n_visits = 500
uids = np.random.choice([f'U{i:03d}' for i in range(1, 101)], size=n_visits)
devices = np.random.choice(['desktop', 'mobile', 'tablet'], size=n_visits, p=[0.5, 0.4, 0.1])

# Generar fechas de inicio dentro de un período de 60 días
start_dates = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 60, size=n_visits), unit='D')
start_times = start_dates + pd.to_timedelta(np.random.randint(0, 86400, size=n_visits), unit='s')

# Duración de sesión entre 5 y 60 minutos
durations = np.random.randint(5, 60, size=n_visits)
end_times = start_times + pd.to_timedelta(durations, unit='m')

source_ids = np.random.choice(['google', 'facebook', 'twitter', 'linkedin'], size=n_visits)

visits_synth = pd.DataFrame({
    'Uid': uids,
    'Device': devices,
    'Start Ts': start_times,
    'End Ts': end_times,
    'Source Id': source_ids
})

print("Visitas (Datos Sintéticos)")
display(visits_synth.head())

# Generar dataset sintético para 'orders'
n_orders = 200
uids_orders = np.random.choice(visits_synth['Uid'].unique(), size=n_orders)

# Para cada pedido, generar una fecha de compra posterior a la primera visita de ese usuario
buy_dates = []
revenues = []
for uid in uids_orders:
    # Seleccionar la primera visita del usuario en el dataset sintético
    first_visit = visits_synth.loc[visits_synth['Uid'] == uid, 'Start Ts'].min()
    # Generar la fecha de compra entre 0 y 10 días después de la primera visita
    delta_days = np.random.randint(0, 11)
    buy_date = first_visit + pd.Timedelta(days=delta_days, seconds=np.random.randint(0, 86400))
    buy_dates.append(buy_date)
    # Generar un revenue aleatorio entre $20 y $200
    revenues.append(np.random.uniform(20, 200))

orders_synth = pd.DataFrame({
    'Uid': uids_orders,
    'Buy Ts': pd.to_datetime(buy_dates),
    'Revenue': revenues
})

print("Orders (Datos Sintéticos)")
display(orders_synth.head())

# Generar dataset sintético para 'costs'
n_costs = 50
dates_cost = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 60, size=n_costs), unit='D')
sources_cost = np.random.choice(['google', 'facebook', 'twitter', 'linkedin'], size=n_costs)
cost_values = np.random.uniform(100, 1000, size=n_costs)

costs_synth = pd.DataFrame({
    'source_id': sources_cost,
    'dt': dates_cost,
    'costs': cost_values
})

print("Costs (Datos Sintéticos)")
display(costs_synth.head())

## 4. Análisis y Métricas

A continuación se aplica el análisis tanto a los datos reales (si se desean usar) como a los sintéticos. En este ejemplo se procederá a utilizar los datasets cargados inicialmente, pero el mismo análisis es aplicable a los datos sintéticos (por ejemplo, reemplazando `visits` por `visits_synth`, etc.).

### 4.1. Análisis de Visitas

En esta sección se calculan:

- **Usuarios únicos:** Cantidad de usuarios únicos que visitan el sitio cada día, semana y mes.
- **Sesiones diarias:** Número total de sesiones por día.
- **Duración de cada sesión:** Diferencia entre *Start Ts* y *End Ts* (en minutos).
- **Frecuencia de retorno:** Tiempo transcurrido entre sesiones consecutivas para cada usuario.

In [ ]:
### Usando el dataset real (puedes cambiar a los sintéticos cambiando la variable, p.ej. visits_synth)

df_visits = visits.copy()  # Cambiar a visits_synth si se quiere analizar el dataset sintético

# Calcular la duración de la sesión en minutos
df_visits['session_duration'] = (df_visits['End Ts'] - df_visits['Start Ts']).dt.total_seconds() / 60

# Usuarios únicos por día
df_visits['date'] = df_visits['Start Ts'].dt.date
daily_users = df_visits.groupby('date')['Uid'].nunique()

# Número de sesiones por día
daily_sessions = df_visits.groupby('date')['Uid'].count()

print("Usuarios únicos diarios:")
display(daily_users.head())

print("Sesiones diarias:")
display(daily_sessions.head())

# Usuarios únicos semanales y mensuales
df_visits['week'] = df_visits['Start Ts'].dt.to_period('W').apply(lambda r: r.start_time.date())
weekly_users = df_visits.groupby('week')['Uid'].nunique()

df_visits['month'] = df_visits['Start Ts'].dt.to_period('M').astype(str)
monthly_users = df_visits.groupby('month')['Uid'].nunique()

print("Usuarios únicos semanales:")
display(weekly_users.head())

print("Usuarios únicos mensuales:")
display(monthly_users.head())

# Distribución de la duración de las sesiones
plt.figure(figsize=(10, 5))
plt.hist(df_visits['session_duration'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribución de la duración de las sesiones (min)')
plt.xlabel('Duración (min)')
plt.ylabel('Frecuencia')
plt.show()

# Calcular la frecuencia de retorno para cada usuario
df_visits_sorted = df_visits.sort_values(by=['Uid', 'Start Ts'])
df_visits_sorted['prev_visit'] = df_visits_sorted.groupby('Uid')['Start Ts'].shift(1)
df_visits_sorted['days_since_last'] = (df_visits_sorted['Start Ts'] - df_visits_sorted['prev_visit']).dt.days

display(df_visits_sorted[['Uid', 'Start Ts', 'prev_visit', 'days_since_last']].head(10))

# Histograma de días entre visitas (sin valores nulos)
plt.figure(figsize=(10, 5))
plt.hist(df_visits_sorted['days_since_last'].dropna(), bins=30, color='salmon', edgecolor='black')
plt.title('Frecuencia de retorno: Días entre visitas')
plt.xlabel('Días entre visitas')
plt.ylabel('Ocurrencias')
plt.show()

### 4.2. Análisis de Ventas

Este apartado se enfoca en:

- **Tiempo hasta la conversión:** Diferencia entre la primera visita y el primer pedido para cada usuario, permitiendo categorizar en cohortes (por ejemplo, Conversion 0d, Conversion 1d, etc.).
- **Cantidad de pedidos y tamaño promedio de compra:** Se agrupan los pedidos por día y se calcula la media del ingreso para evaluar el tamaño de compra.
- **Lifetime Value (LTV):** Suma total de ingresos generados por usuario.

In [ ]:
# Extraer la primera visita (registro) de cada usuario
first_visit = df_visits.groupby('Uid')['Start Ts'].min().reset_index().rename(columns={'Start Ts': 'first_visit'})

# Extraer el primer pedido de cada usuario
first_order = orders.groupby('Uid')['Buy Ts'].min().reset_index().rename(columns={'Buy Ts': 'first_order'})

# Unir ambas tablas para calcular el tiempo hasta conversión
conversion = pd.merge(first_visit, first_order, on='Uid', how='inner')
conversion['conversion_days'] = (conversion['first_order'] - conversion['first_visit']).dt.days

print("Tiempo hasta la conversión (días) para algunos usuarios:")
display(conversion[['Uid', 'first_visit', 'first_order', 'conversion_days']].head(10))

# Histograma de los días hasta la conversión
plt.figure(figsize=(10, 5))
plt.hist(conversion['conversion_days'], bins=range(0, conversion['conversion_days'].max()+2), color='lightgreen', edgecolor='black', align='left')
plt.title('Días hasta la conversión')
plt.xlabel('Días')
plt.ylabel('Cantidad de usuarios')
plt.show()

# Pedidos por día
orders['order_date'] = orders['Buy Ts'].dt.date
orders_per_day = orders.groupby('order_date').size()

plt.figure(figsize=(12, 5))
orders_per_day.plot(kind='line', marker='o')
plt.title('Número de Pedidos por Día')
plt.xlabel('Fecha')
plt.ylabel('Número de Pedidos')
plt.xticks(rotation=45)
plt.show()

# Tamaño promedio de compra
average_order_value = orders['Revenue'].mean()
print(f"Tamaño promedio de compra: ${average_order_value:.2f}")

# Lifetime Value (LTV) por usuario
ltv_per_user = orders.groupby('Uid')['Revenue'].sum()
print("Ejemplo de LTV por usuario:")
display(ltv_per_user.head(10))

### 4.3. Análisis de Marketing

Se analizan los siguientes aspectos:

1. **Gasto total en publicidad:** Tanto global como segmentado por *source_id*.
2. **Costo de adquisición de clientes (CAC):** Gasto dividido entre el número de usuarios convertidos, usando la fuente de adquisición que aparece en la primera visita.
3. **Rentabilidad (ROMI):** Cálculo del retorno de la inversión basado en los ingresos generados por fuente y el gasto asociado.

Además, se generan gráficos para visualizar estas métricas y se segmenta la información por dispositivo.

In [ ]:
# Gasto total en marketing
total_cost = costs['costs'].sum()
print(f"Gasto total en publicidad: ${total_cost:.2f}")

# Gasto por fuente
cost_by_source = costs.groupby('source_id')['costs'].sum()
print("Gasto por fuente de adquisición:")
display(cost_by_source)

# Graficar gasto por fuente (barras)
plt.figure(figsize=(10,5))
cost_by_source.plot(kind='bar', edgecolor='black')
plt.title('Gasto en Publicidad por Fuente')
plt.xlabel('Fuente')
plt.ylabel('Costos')
plt.show()

### Cálculo del Costo de Adquisición de Clientes (CAC)

# Suponemos que la fuente de adquisición de cada usuario es la de su primer registro
user_source = df_visits.sort_values(by='Start Ts').groupby('Uid').first().reset_index()[['Uid', 'Source Id']]

# Unir la información de conversión con la fuente
conversion_with_source = pd.merge(conversion, user_source, on='Uid', how='left')

# Número de clientes adquiridos por fuente
customers_by_source = conversion_with_source.groupby('Source Id')['Uid'].nunique()
print("Clientes adquiridos por fuente:")
display(customers_by_source)

# Calcular el CAC: gasto de la fuente / número de clientes adquiridos
cac = cost_by_source / customers_by_source
print("Costo de Adquisición de Clientes (CAC) por fuente:")
display(cac)

### Cálculo del ROMI (Return on Marketing Investment)

# Asociar órdenes con la fuente de adquisición (utilizando la fuente del primer registro)
orders_with_source = pd.merge(orders, user_source, on='Uid', how='left')

# Ingresos por fuente
revenue_by_source = orders_with_source.groupby('Source Id')['Revenue'].sum()

# Calcular ROMI: (ingresos - gasto) / gasto
romi = (revenue_by_source - cost_by_source) / cost_by_source
print("ROMI por fuente de adquisición:")
display(romi)

# Graficar CAC y ROMI
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

cac.plot(kind='bar', ax=ax[0], edgecolor='black')
ax[0].set_title('Costo de Adquisición de Clientes (CAC)')
ax[0].set_xlabel('Fuente')
ax[0].set_ylabel('Costo')

romi.plot(kind='bar', ax=ax[1], edgecolor='black')
ax[1].set_title('ROMI por Fuente')
ax[1].set_xlabel('Fuente')
ax[1].set_ylabel('ROMI')

plt.tight_layout()
plt.show()

### Análisis adicional: Segmentación por dispositivo

# Número de usuarios únicos diarios por dispositivo
users_by_device = df_visits.groupby([df_visits['Start Ts'].dt.date, 'Device'])['Uid'].nunique().unstack()

plt.figure(figsize=(12, 5))
users_by_device.plot(kind='line', marker='o')
plt.title('Usuarios Únicos Diarios por Dispositivo')
plt.xlabel('Fecha')
plt.ylabel('Número de Usuarios')
plt.xticks(rotation=45)
plt.legend(title='Device')
plt.show()

## 5. Conclusiones y Recomendaciones de Marketing

Tras analizar las métricas, se destacan los siguientes puntos:

- **Tiempo hasta la conversión:** Se observa que un número importante de usuarios realiza la primera compra en 0 o 1 día, lo cual indica que ciertos canales pueden atraer usuarios altamente interesados desde el primer contacto.

- **Costo de Adquisición (CAC):** La comparación del CAC entre las fuentes muestra que algunas permiten adquirir clientes a menor costo. Esto sugiere focalizar la inversión en estos canales.

- **Rentabilidad (ROMI):** Las fuentes con altos ingresos en comparación con el gasto son las más rentables. Las inversiones en canales con ROMI negativo deben ser evaluadas y optimizadas.

- **Segmentación por dispositivo:** Se observan diferencias en el comportamiento de los usuarios según el dispositivo, lo que sugiere adaptar las estrategias de marketing para cada segmento.

### Recomendación Final:

1. **Invertir en fuentes con bajo CAC y alto ROMI:** Esto permitirá optimizar la inversión en publicidad y lograr un mayor retorno.
2. **Impulsar campañas que generen conversiones rápidas:** Dado que una gran parte de los usuarios realiza su primera compra en el mismo día o al día siguiente, es importante enfocarse en este tipo de campañas.
3. **Revisar y ajustar inversiones en canales de bajo rendimiento:** Canalizar esfuerzos y presupuesto hacia aquellos canales que demuestren mejor rendimiento en conversión y rentabilidad.
4. **Adaptar estrategias según el dispositivo:** Los datos indican que el comportamiento varía entre dispositivos, por lo que las campañas pueden beneficiarse al ser personalizadas para cada segmento.

En resumen, recomiendo destinar una mayor inversión en los canales que, en base a las métricas evaluadas (tiempo hasta la conversión, CAC y ROMI), permiten conseguir conversiones rápidas y rentables. Esto favorecerá el escalado de la inversión publicitaria y optimizará el retorno de la inversión en marketing.